# Environment Setup

In [ ]:
! pip install transformers huggingface torch

In [ ]:
! pip install -U "huggingface_hub[cli]"


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# Import Library

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,pipeline

# Load Model

### Ministral

In [ ]:
! pip install vllm

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("mistralai/Ministral-8B-Instruct-2410", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("mistralai/Ministral-8B-Instruct-2410", trust_remote_code=True)


# Dataset processed

## Cloud

### Load

In [ ]:
pip install pandas

In [20]:
import pandas as pd

In [21]:
#from google.colab import drive
#drive.mount('/content/drive')

In [22]:
cloud = pd.read_excel("../../../data/splits/cloud/cloud_test.xlsx")

In [23]:
cloud.drop(['Unnamed: 0'],axis=1,inplace=True)

In [ ]:
cloud.head(2)

### Processing

In [25]:
cloud=cloud[['title','body','selected_answer']]

In [ ]:
cloud.head()

In [ ]:
cloud['context'] = "title: " + cloud['title'].astype(str) + "\nbody: " + cloud['body'].astype(str).str.strip()


In [28]:
cloud_processed=pd.DataFrame(
    {
        'context':cloud['context'],
        'gt':cloud['selected_answer']
    }
)

In [ ]:
cloud_processed.head(1)

In [ ]:
cloud_processed[cloud_processed.duplicated()].sum()

## Device

### Load

In [ ]:
import pandas as pd

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
device = pd.read_excel("../../../data/splits/device/device_test.xlsx")

In [ ]:
#device.drop(['Unnamed: 0'],axis=1,inplace=True)

In [ ]:
device.head(2)

### Processing

In [ ]:
device=cloud[['title','body','selected_answer']]

In [ ]:
device.head()

In [ ]:
device['context'] = "title: " + device['title'].astype(str) + "\nbody: " + device['body'].astype(str).str.strip()


In [ ]:
device_processed=pd.DataFrame(
    {
        'context':device['context'],
        'gt':device['selected_answer']
    }
)

In [ ]:
device_processed.head(1)

In [ ]:
device_processed[device_processed.duplicated()].sum()

# Prompt

## Cloud

### Zero Shot

In [31]:
zero_shot = '''
You are an expert cloud assistant. Provide a concise answer to the following question.
If you lack sufficient information to answer, respond with "No answer."

Question:
{}

Answer:

'''

In [32]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['context'])

cloud_processed['zero-shot-question'] = cloud_processed.apply(generate_zero_shot_prompt, axis=1)


In [ ]:
print(cloud_processed['zero-shot-question'][0])

### CoT

In [34]:
cot = '''
You are an expert cloud assistant. Provide a concise answer to the following question.
If you lack sufficient information to answer, respond with "No answer."
Question:
{}

Lets think step by step:
Answer:
'''

In [35]:
def generate_cot_prompt(row):
    return cot.format(row['context'])

cloud_processed['cot-question'] = cloud_processed.apply(generate_cot_prompt, axis=1)


In [ ]:
print(cloud_processed['cot-question'][0])

In [ ]:
cloud_processed.info()

## Device

### Zero Shot

In [ ]:
zero_shot = '''
You are an expert device assistant. Provide a concise answer to the following question.
If you lack sufficient information to answer, respond with "No answer."

Question:
{}

Answer:

'''

In [ ]:
def generate_zero_shot_prompt(row):
    return zero_shot.format(row['context'])

device_processed['zero-shot-question'] =device_processed.apply(generate_zero_shot_prompt, axis=1)


In [ ]:
print(device_processed['zero-shot-question'][0])

### CoT

In [ ]:
cot = '''
You are an expert cloud assistant. Provide a concise answer to the following question.
If you lack sufficient information to answer, respond with "No answer."
Question:
{}

Lets think step by step:
Answer:
'''

In [ ]:
def generate_cot_prompt(row):
    return cot.format(row['context'])

device_processed['cot-question'] = device_processed.apply(generate_cot_prompt, axis=1)


In [ ]:
print(device_processed['cot-question'][0])

In [ ]:
device_processed.info()

# Generate Response

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# Zero Shot Generated Response
def generate_zero_shot_answer(row, tokenizer, model, max_length=768):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    if 'zero-shot-question' not in row:
        raise ValueError("Input row must contain the key 'zero-shot-question'")
    
    input_ids = tokenizer(row['zero-shot-question'], return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        output = model.generate(input_ids, max_length=max_length, synced_gpus=False)
    return tokenizer.decode(output[0], skip_special_tokens=True)


In [ ]:
# CoT Generated Response

def generate_cot_answer(row, tokenizer, model, max_length=768):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model.to(device)
    
    if 'cot-question' not in row:
        raise ValueError("Input row must contain the key 'cot-question'")
    
    input_ids = tokenizer(row['cot-question'], return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        output = model.generate(input_ids, max_length=max_length, synced_gpus=False) 
    return tokenizer.decode(output[0], skip_special_tokens=True)


## Sample

In [37]:
sample=cloud_processed[:4]

In [ ]:
sample.head(2)

In [ ]:
sample['ministral-zero-shot-answer'] = sample.apply(
    lambda row: generate_zero_shot_answer(row, tokenizer, model), 
    axis=1
)

In [ ]:
print(sample['ministral-zero-shot-answer'][0])

In [ ]:
sample['ministral-cot-answer'] = sample.apply(lambda row: generate_cot_answer(row, tokenizer, model), axis=1)

In [ ]:
print(sample['ministral-cot-answer'][0])

In [ ]:
sample.to_csv("sample_llm_text_mistral_8b_instruct_2410.csv")

## Full test set

### Cloud Testset

In [ ]:
cloud_testset=cloud_processed

In [ ]:
cloud_testset['ministral-zero-shot-answer'] = cloud_testset.apply(lambda row: generate_zero_shot_answer(row, tokenizer, model), axis=1)

In [ ]:
cloud_testset.head(2)

In [ ]:
cloud_testset['ministral-cot-answer'] = cloud_testset.apply(lambda row: generate_cot_answer(row, tokenizer, model), axis=1)

In [ ]:
cloud_testset['ministral-cot-answer'].head(2)

In [ ]:
cloud_testset.to_excel("../../../outputs/responses/baselines/llm_text/cloud/llm_text_response_cloud_mistral_8b_instruct_2410_zero_shot.xlsx", index=False)

### Device Test Set

In [ ]:
device_testset=device_processed

In [ ]:
device_testset['ministral-zero-shot-answer'] = device_testset.apply(lambda row: generate_zero_shot_answer(row, tokenizer, model), axis=1)

In [ ]:
device_testset.head(2)

In [ ]:
device_testset['ministral-cot-answer'] = device_testset.apply(lambda row: generate_cot_answer(row, tokenizer, model), axis=1)

In [ ]:
device_testset['ministral-cot-answer'].head(2)

In [ ]:
cloud_testset.to_excel("../../../outputs/responses/baselines/llm_text/device/llm_text_response_device_mistral_8b_instruct_2410_zero_shot.xlsx", index=False)